# Notebook B4: conflict trials, and when do the *gain loss* and the *prior offset* arrive? (GPU, about 50 min)

**Where we are.** On cached data (no GPU) a two-number rule explains the history effect on extreme-count trials: $\Delta_{hist} \approx 0.43\,\Delta_{none} + 0.63\,\mathrm{sign}(\text{count}) - 0.11$, i.e. the history sentence (i) shrinks the evidence part of $\Delta$ by about 0.4 (**gain**) and (ii) adds a push toward the majority side (**offset**). Both groups, congruent and conflict trials, fit it equally well. From the existing patching rows, the gain falls at layers 12-16 while the offset only appears at layers 16-20.

**What this notebook adds**
1. **More trials.** B patched only 19 conflict trials (count points against the stimulus). Here we patch *all* remaining conflict trials plus 100 more congruent ones, same recipe as B1 (history state into a no-history run and back, at the last position). Combined with B's rows this gives 71 conflict and about 190 congruent trials.
2. **Gain and offset per layer.** At each layer $L$ we fit $\Delta_{patched}(L) = a_L\,\Delta_{none} + c_L\,\mathrm{sign} + b_L$, so each layer gives how much of the gain loss and how much of the offset the last-position state already carries. Finer layer grid (10-24).
3. **Conflict vs congruent**, separately, with the same decomposition and with answer flips.

**Setup:** GPU on (2x T4), Internet on, dataset `llava-pve-outputs-run` attached, Kaggle secret `KAGGLE_KEY` on. *Save & Run All (Commit)*. Checkpoints to `ponishbhatia3/llava-pve-b4-ckpt`; resumes if the session dies.
**Tested on a mock model, not on the real LLaVA; the self-test cell stops with a clear message if a hook is wrong.**

In [ ]:
# ===================== CONFIG =====================
MOCK = False
QUICK = False                # True = tiny smoke test (~6 min)
SEED = 0
DATA = "/kaggle/input/datasets/ponishbhatia3/llava-pve-outputs-run"
OUT = "/kaggle/working/B4_out"
N_CONG = 100                                 # extra congruent extreme-count trials (all remaining conflict trials are used)
PATCH_LAYERS = [10, 12, 13, 14, 15, 16, 17, 18, 19, 20, 22, 24]
DIRS = ["shift", "prior", "evid", "shuf", "rand1"]          # needed by the shared direction code
B_DONE_T = set([21, 25, 26, 41, 43, 45, 49, 50, 52, 55, 59, 65, 66, 69, 71, 79, 80, 81, 87, 88, 91, 99, 103, 108, 110, 112, 113, 116, 122, 123, 124, 125, 133, 137, 144, 145, 149, 158, 162, 166, 168, 169, 170, 177, 179, 183, 185, 187, 188, 191, 192, 194, 197, 201, 206, 212, 213, 214, 219, 220, 221, 223, 224, 228, 229, 233, 234, 236, 237, 240, 241, 243, 260, 269, 274, 275, 277, 285, 286, 287, 288, 289, 294, 295, 305, 309, 312, 316, 320, 331, 334, 335, 342, 345, 347, 355, 358, 359, 361, 362, 366, 373, 374, 375, 376, 377, 380, 381, 383, 389, 393, 396, 397, 400, 407, 410, 412, 418, 420, 425, 429, 430, 432, 433, 437, 443, 447, 449, 453, 463, 466, 469, 470, 478, 479, 480, 481, 483, 485, 498, 507, 510, 518, 520, 523, 528, 529, 533, 548, 551, 560, 562, 569, 571, 572, 573, 575, 582, 596, 599])   # the 160 trials already patched in notebook B (not repeated here)
if QUICK:
    N_CONG, PATCH_LAYERS = 8, [12, 16, 20]
import os, re, glob, json, math, time, zipfile, shutil, warnings
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from PIL import Image
warnings.filterwarnings("ignore")
os.makedirs(OUT, exist_ok=True)
SUMMARY = []
def say(s=""): print(s); SUMMARY.append(s)

In [ ]:
# ===================== CHECKPOINTS TO A KAGGLE DATASET (so a dead session loses nothing) =====================
KAGGLE_USER = "ponishbhatia3"
CKPT_SLUG = f"{KAGGLE_USER}/llava-pve-b4-ckpt"          # private dataset, created on the first push, new version afterwards
PUSH = not MOCK                                         # set False to disable (then only /kaggle/working is saved)
PUSH_EVERY_MIN = 10
CKPT_DIR = "/kaggle/working/B4_ckpt"
RUN_CFG = dict(SEED=SEED, QUICK=QUICK, N_CONG=N_CONG, PATCH_LAYERS=PATCH_LAYERS)
json.dump(RUN_CFG, open(f"{OUT}/run_config.json", "w"))
_last_push = 0.0

def _setup_creds():
    assert KAGGLE_USER and "YOUR" not in KAGGLE_USER.upper(), "KAGGLE_USER is a placeholder"
    from kaggle_secrets import UserSecretsClient
    sc, key = UserSecretsClient(), None
    for name in ("KAGGLE_KEY", "KAGGLE_API_TOKEN"):
        try: key = sc.get_secret(name); break
        except Exception: pass
    assert key, "No Kaggle secret found. Add-ons -> Secrets -> add KAGGLE_KEY (your Kaggle API key / token from Settings -> API), switch it ON for this notebook, rerun. Or set PUSH = False."
    os.environ["KAGGLE_USERNAME"] = KAGGLE_USER
    os.environ["KAGGLE_API_TOKEN" if key.startswith("KGAT_") else "KAGGLE_KEY"] = key

def _api():
    from kaggle.api.kaggle_api_extended import KaggleApi
    a = KaggleApi(); a.authenticate(); return a

def push_ckpt(note):
    os.makedirs(CKPT_DIR, exist_ok=True)
    open(f"{CKPT_DIR}/README.txt", "w").write("Checkpoints of notebook B4 (conflict-trial patching). Resume by attaching this dataset as input and rerunning with the same settings.\n")
    for pat in ("*.csv", "*.png", "*.txt", "*.json"):
        for f in glob.glob(f"{OUT}/{pat}"): shutil.copy(f, CKPT_DIR)
    json.dump({"title": "LLaVA PVE B4 checkpoints", "id": CKPT_SLUG, "licenses": [{"name": "CC0-1.0"}]}, open(f"{CKPT_DIR}/dataset-metadata.json", "w"))
    api = _api()
    try: api.dataset_status(CKPT_SLUG); exists = True
    except Exception: exists = False
    res = api.dataset_create_version(CKPT_DIR, note, quiet=True, dir_mode="zip") if exists else api.dataset_create_new(CKPT_DIR, public=False, quiet=True, dir_mode="zip")
    err = getattr(res, "error", None)
    assert not err, f"Kaggle returned an error: {err}"
    print(f"[checkpoint] {'new version' if exists else 'created dataset'} {CKPT_SLUG} ({note}) | server says: {getattr(res, 'status', res)}", flush=True)

def maybe_push(note, force=False):
    global _last_push
    if not PUSH or (not force and time.time() - _last_push < PUSH_EVERY_MIN * 60): return
    try: push_ckpt(note); _last_push = time.time()
    except Exception as e: print(f"!!! CHECKPOINT PUSH FAILED (the run continues; files are still in {OUT}): {type(e).__name__}: {str(e)[:300]}", flush=True)

# resume: if a previous checkpoint dataset is attached as input with the SAME settings, copy its files back
for hit in glob.glob("/kaggle/input/**/run_config.json", recursive=True):
    try:
        if "llava-pve-outputs-run" in hit or json.load(open(hit)) != json.loads(json.dumps(RUN_CFG)): continue
    except Exception: continue
    d = os.path.dirname(hit)
    for name in ("patch4_rows.csv",):
        if os.path.exists(f"{d}/{name}") and not os.path.exists(f"{OUT}/{name}"): shutil.copy(f"{d}/{name}", f"{OUT}/{name}"); print("resumed", name, "from", d)

if PUSH:                                                # fail fast: a real test push now, not after 40 minutes
    _setup_creds(); push_ckpt("preflight")
    print("checkpoint push works; later pushes happen every", PUSH_EVERY_MIN, "min and at the end of each stage")

In [ ]:
# ===================== trials, prompts, images: identical to the main run (counts10 history) =====================
CONTRASTS = [0.0, 0.03, 0.045, 0.0625, 0.09, 0.125, 1.0]; N_FULL, FULL_UNBIASED, WINDOW = 600, 90, 10
SIZE = 336; _Y, _X = np.mgrid[0:SIZE, 0:SIZE].astype(np.float32)
def make_trials(n, seed, contrasts, n_unbiased):
    rng = np.random.default_rng(seed); p = np.full(n, 0.5)
    t, cur = n_unbiased, float(rng.choice([0.2, 0.8]))
    while t < n:
        L = 20 + int(rng.exponential(60))
        while L > 100: L = 20 + int(rng.exponential(60))
        p[t:t + L] = cur; t, cur = t + L, round(1.0 - cur, 2)
    side = np.where(rng.random(n) < p, "right", "left"); c = rng.choice(contrasts, n)
    return dict(n=n, p_right=p, side=side, contrast=c, dx=rng.uniform(-15, 15, n), dy=rng.uniform(-30, 30, n), phi=rng.uniform(0, 2 * np.pi, n))
def render(T, t, seed):
    cx = SIZE / 2 + (84 if T["side"][t] == "right" else -84) + T["dx"][t]; cy = SIZE / 2 + T["dy"][t]
    env = np.exp(-((_X - cx) ** 2 + (_Y - cy) ** 2) / (2 * 36 ** 2))
    a = 0.5 + 0.5 * T["contrast"][t] * env * np.sin(2 * np.pi * (_X - cx) / 24 + T["phi"][t])
    a = a + np.random.default_rng(seed * 100003 + t).normal(0, 0.02, a.shape)
    return Image.fromarray(np.stack([(np.clip(a, 0, 1) * 255).astype(np.uint8)] * 3, -1))
def hist_counts(T, t, w):
    past = T["side"][max(0, t - w):t]; return len(past), int((past == "right").sum())
def history_text(T, t):
    n, nr = hist_counts(T, t, WINDOW)
    return "No previous trials." if n == 0 else f"In the last {n} trials the grating was on the left {n - nr} times and on the right {nr} times."
QS = ["Is the grating on the left or right of the image? Answer with one word.", "Is the grating on the right or left of the image? Answer with one word."]
def prompt(hist, q): return f"USER: <image>\n{hist}\n{q} ASSISTANT:" if hist else f"USER: <image>\n{q} ASSISTANT:"
T = make_trials(N_FULL, SEED + 1, CONTRASTS, FULL_UNBIASED)

In [ ]:
# ===================== cache + directions (recomputed from the cached residual streams, same recipe as notebook A) =====================
ROWS = pd.read_csv(glob.glob(f"{DATA}/**/full_rows_all.csv", recursive=True)[0])
HS = np.concatenate([np.load(f, mmap_mode="r") for f in sorted(glob.glob(f"{DATA}/**/full_hs_*.npy", recursive=True))])    # (trial, [none,hist], layer 0..32, dim)
NTR, _, NLAY, D = HS.shape
w_ = ROWS.pivot(index="t", columns="order", values="delta")
M = ROWS[ROWS.order == "none"].set_index("t").loc[w_.index][["side", "contrast", "p_right", "n_hist", "n_hist_right"]].copy()
M["d_none"], M["d_hist"] = w_["none"], w_["hist"]; M["shift"] = M.d_hist - M.d_none
assert (M.side.values == T["side"]).all() and np.allclose(M.contrast.values, T["contrast"]), "regenerated trials differ from the main run: check SEED / CONTRASTS"
NR = M.n_hist_right.values.astype(int); FULL = (M.n_hist.values == 10)
frac = (NR + .5) / (M.n_hist.values + 1); TP = np.log(frac / (1 - frac))
cell = pd.get_dummies(M.side.astype(str) + "|" + M.contrast.astype(str), dtype=float).values
lsq = lambda A, B: np.linalg.lstsq(A, B, rcond=None)[0]
def count_dir(X, target, Cc):
    Xr = X - Cc @ lsq(Cc, X); yt = target - Cc @ lsq(Cc, target); d = Xr.T @ yt; return d / (np.linalg.norm(d) + 1e-12)
hi_c = M.contrast.values >= 0.09; sr = M.side.values == "right"
hi_cnt, lo_cnt = FULL & (NR >= 8), FULL & (NR <= 2)
rng0 = np.random.default_rng(0); perm = rng0.permutation(FULL.sum())
DIRVEC, NORM, RESNORM = {}, np.zeros(NLAY), np.zeros(NLAY)
for name in DIRS: DIRVEC[name] = np.zeros((NLAY, D), np.float32)
UNIT = {name: np.zeros((NLAY, D), np.float32) for name in ["shift", "prior", "evid", "shuf", "rand1", "rand2"]}    # unit-norm versions, always computed
for L in range(1, NLAY):
    Xn, Xh = HS[:, 0, L, :].astype(np.float32), HS[:, 1, L, :].astype(np.float32)
    shift = ((Xh - Xn)[hi_cnt].mean(0) - (Xh - Xn)[lo_cnt].mean(0)) / 2          # natural half-jump from a no-history state to a high/low-count state
    NORM[L] = np.linalg.norm(shift); RESNORM[L] = np.linalg.norm(Xn, axis=1).mean(); unit = lambda v: v / (np.linalg.norm(v) + 1e-12)
    cand = {"shift": unit(shift),
            "prior": count_dir(Xh[FULL], TP[FULL], cell[FULL]),
            "evid": unit(Xn[hi_c & sr].mean(0) - Xn[hi_c & ~sr].mean(0)),
            "shuf": count_dir(Xh[FULL], TP[FULL][perm], cell[FULL]),
            "rand1": unit(np.random.default_rng(11 + L).normal(size=D)), "rand2": unit(np.random.default_rng(97 + L).normal(size=D))}
    for name in cand: UNIT[name][L] = cand[name]
    for name in DIRS: DIRVEC[name][L] = cand[name] * NORM[L]                     # every direction gets the SAME norm
nat_hi = M["shift"][hi_cnt].mean(); nat_lo = M["shift"][lo_cnt].mean(); NAT_HALF = (nat_hi - nat_lo) / 2
print(f"cache {HS.shape} | natural behavioral half-jump in delta (high-count minus low-count)/2 = {NAT_HALF:.2f} logits")
print("natural half-jump norm | typical residual norm, per layer:", {L: (round(float(NORM[L]), 2), round(float(RESNORM[L]), 1)) for L in (8, 10, 12, 14, 16, 20, 24)})
print("cos(count direction, evidence direction) at layers 8/12/16/20:", {L: round(float(UNIT['prior'][L] @ UNIT['evid'][L]), 2) for L in (8, 12, 16, 20)})

In [ ]:
# ===================== model runner =====================
if MOCK:
    class Runner:
        """Numpy stand-in: state at layer L is linear in (evidence, count); delta reads it out. Pipeline test only."""
        def __init__(s): s.rng = np.random.default_rng(5)
        def prep(s, img, text):
            a = np.abs(np.asarray(img)[..., 0].astype(float) - 128); cx = (a.sum(0) * np.arange(SIZE)).sum() / (a.sum() + 1e-6)
            m = re.search(r"on the right (\d+) times", text); n = None
            if m: nr = int(m.group(1)); nl = int(re.search(r"on the left (\d+) times", text).group(1)); n = (nr + .5) / (nr + nl + 1)
            return dict(ev=float(np.clip((cx - 168) * 0.03, -1.7, 1.7)), frac=n)
        def state(s, inp, L):
            x = np.zeros(D, np.float32); h = np.random.default_rng(int(abs(inp["ev"]) * 1e4) % 99991)
            x += 0.02 * h.normal(size=D)
            if L >= 4: x[0] += inp["ev"] * (0.5 if inp["frac"] is not None else 1.0)
            if L >= 8 and inp["frac"] is not None: x[1] += (inp["frac"] - .5) * 3; x[0] += (inp["frac"] - .5) * 0.5
            return x
        def readout(s, x): return -0.7 + 3.0 * x[0] + 1.0 * x[1]
        def delta(s, inp, edit=None):
            x = s.state(inp, 32)
            if edit is not None:
                L, mode, vec = edit; xl = s.state(inp, L)
                x = x + ((vec - xl) if mode == "replace" else vec)
            return float(s.readout(x))
        def hs(s, inp): return np.stack([s.state(inp, L) for L in range(NLAY)]).astype(np.float16)
    RUN = Runner()
else:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes", "accelerate"], check=False)
    import torch
    from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
    NAME = "llava-hf/llava-1.5-7b-hf"
    model = LlavaForConditionalGeneration.from_pretrained(NAME, torch_dtype=torch.float16, device_map="auto",
                quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)).eval()
    proc = AutoProcessor.from_pretrained(NAME)
    L_ID, R_ID = [proc.tokenizer.encode(w, add_special_tokens=False)[0] for w in ("Left", "Right")]
    _lists = [(n, m) for n, m in model.named_modules() if isinstance(m, torch.nn.ModuleList) and len(m) == 32]
    assert len(_lists) == 1, f"expected exactly one 32-layer list, found {[n for n, _ in _lists]}"
    LAYERS = _lists[0][1]; print("language-model layers found at:", _lists[0][0])
    class Runner:
        def prep(s, img, text): return proc(images=img, text=text, return_tensors="pt").to(model.device, torch.float16)
        def delta(s, inp, edit=None):
            handle = None
            if edit is not None:
                L, mode, vec = edit; v0 = torch.tensor(vec, dtype=torch.float16)
                def hook(mod, args, output):                      # output of decoder layer L-1 == hidden_states[L]
                    h = output[0] if isinstance(output, tuple) else output
                    v = v0.to(h.device, h.dtype)                  # the model can be split over 2 GPUs: put the vector where this layer lives
                    h = h.clone()
                    if mode == "replace": h[:, -1, :] = v
                    else: h[:, -1, :] = h[:, -1, :] + v
                    return (h,) + tuple(output[1:]) if isinstance(output, tuple) else h
                handle = LAYERS[L - 1].register_forward_hook(hook)
            try:
                with torch.no_grad(): lg = model(**inp).logits[0, -1].float()
            finally:
                if handle is not None: handle.remove()
            return float((lg[R_ID] - lg[L_ID]).item())
        def hs(s, inp):
            with torch.no_grad(): out = model(**inp, output_hidden_states=True)
            return torch.stack([h[0, -1] for h in out.hidden_states]).float().cpu().numpy().astype(np.float16)
        def captured(s, inp, L):                                  # what the hook sees at layer L (for the self-test)
            box = {}
            def hook(mod, args, output): box["h"] = (output[0] if isinstance(output, tuple) else output)[0, -1].float().cpu().numpy()
            hd = LAYERS[L - 1].register_forward_hook(hook)
            try:
                with torch.no_grad(): model(**inp)
            finally: hd.remove()
            return box["h"]
    RUN = Runner()
print("runner ready")

In [ ]:
# ===================== SELF-TEST: stop here if anything is off =====================
def inputs(t, with_hist, q=0):
    return RUN.prep(render(T, t, SEED + 1), prompt(history_text(T, t) if with_hist else None, QS[q]))
ok = np.where(FULL & (M.contrast.values >= 0.09))[0]; test_ts = list(ok[:6])
res = []
for t in test_ts:
    dn = (RUN.delta(inputs(t, False, 0)) + RUN.delta(inputs(t, False, 1))) / 2; dh = (RUN.delta(inputs(t, True, 0)) + RUN.delta(inputs(t, True, 1))) / 2
    res.append((dn - M.d_none.values[t], dh - M.d_hist.values[t]))
res = np.array(res); say(f"[self-test 1] reproduction of the cached delta (new - cached): none max|diff| {np.abs(res[:, 0]).max():.3f}, hist max|diff| {np.abs(res[:, 1]).max():.3f}")
assert np.abs(res).max() < 0.35, "cannot reproduce the cached deltas: trials, prompts or model setup differ from the main run"
t = test_ts[0]; inp = inputs(t, True, 0); d0 = RUN.delta(inp)
d_id = RUN.delta(inp, edit=(12, "add", np.zeros(D, np.float32))); say(f"[self-test 2] adding a zero vector changes delta by {abs(d_id - d0):.4f} (should be ~0)")
assert abs(d_id - d0) < 0.05
own = HS[t, 1, 12, :].astype(np.float32); d_own = RUN.delta(inp, edit=(12, "replace", own)); say(f"[self-test 3] replacing the state with the history run's own cached state changes delta by {abs(d_own - d0):.4f} (should be small)")
assert abs(d_own - d0) < 0.25
if not MOCK:
    cap = RUN.captured(inp, 12); cache = HS[t, 1, 12, :].astype(np.float32)
    cs = float(cap @ cache / (np.linalg.norm(cap) * np.linalg.norm(cache))); say(f"[self-test 4] hook state vs cached hidden_states[12]: cosine {cs:.4f} (should be ~1)")
    assert cs > 0.98, "hook indexing does not match hidden_states: layer off by one?"
Lt = 20                                                  # layer 20 lives on the SECOND GPU: this also tests the cross-device 'add' hook
big_vec = (UNIT["evid"][Lt] * 0.5 * RESNORM[Lt]).astype(np.float32); big = RUN.delta(inp, edit=(Lt, "add", big_vec))
say(f"[self-test 5] adding the evidence direction (half the typical residual norm) at layer {Lt} moves delta by {big - d0:+.2f} (should be clearly positive)")
assert big - d0 > (0.01 if MOCK else 0.5), "steering along the evidence direction does nothing: the 'add' hook is broken"
say("self-tests passed")

In [ ]:
# ===================== trial selection + patching (checkpointed) =====================
sgn_all = np.where(NR >= 5, 1.0, -1.0)
EXT = FULL & ((NR >= 8) | (NR <= 2)); CONF = EXT & (((NR >= 8) & (M.side.values == "left")) | ((NR <= 2) & (M.side.values == "right")))
rng = np.random.default_rng(7)
conf_t = [t for t in np.where(CONF)[0] if t not in B_DONE_T]
cong_pool = [t for t in np.where(EXT & ~CONF)[0] if t not in B_DONE_T]
cong_t = list(rng.choice(cong_pool, min(N_CONG, len(cong_pool)), replace=False))
if QUICK: conf_t = conf_t[:8]
PATCH_T = np.sort(np.array(conf_t + cong_t, dtype=int))
say(f"extreme-count full-window trials in the main run: {EXT.sum()} (conflict {CONF.sum()}, congruent {(EXT & ~CONF).sum()}); new trials to patch: conflict {len(conf_t)}, congruent {len(cong_t)}")
pp = f"{OUT}/patch4_rows.csv"; done = pd.read_csv(pp) if os.path.exists(pp) else pd.DataFrame()
have = set(done.t) if len(done) else set(); rows = done.to_dict("records") if len(done) else []
t0 = time.time(); todo = [t for t in PATCH_T if t not in have]; print(f"to do: {len(todo)} trials x {2 + 2 * len(PATCH_LAYERS)} forwards")
for i, t in enumerate(todo):
    inn, inh = inputs(t, False, 0), inputs(t, True, 0)
    rec = dict(t=int(t), side=M.side.values[t], contrast=M.contrast.values[t], nr=int(NR[t]), conflict=bool(CONF[t]), d_none=RUN.delta(inn), d_hist=RUN.delta(inh))
    for L in PATCH_LAYERS:
        rec[f"fwd_{L}"] = RUN.delta(inn, edit=(L, "replace", HS[t, 1, L, :].astype(np.float32)))              # history state -> no-history run
        rec[f"rev_{L}"] = RUN.delta(inh, edit=(L, "replace", HS[t, 0, L, :].astype(np.float32)))              # no-history state -> history run
    rows.append(rec)
    if (i + 1) % 10 == 0 or i == len(todo) - 1:
        pd.DataFrame(rows).to_csv(pp, index=False); maybe_push("patch4"); el = time.time() - t0
        print(f"  {i + 1}/{len(todo)} | {el / 60:.1f} min | ~{el / (i + 1) * (len(todo) - i - 1) / 60:.0f} min left", flush=True)
PT = pd.DataFrame(rows).sort_values("t").reset_index(drop=True); PT.to_csv(pp, index=False); maybe_push("patch4 done", force=True)

In [ ]:
# ===================== analysis 1: behavior on ALL extreme-count trials of the main run (cached deltas, no model) =====================
rb = np.random.default_rng(0)
def fitgo(y, dn, sg, extra=None):
    X = np.c_[dn, sg, np.ones(len(dn))] if extra is None else np.c_[dn, sg, np.ones(len(dn)), extra]; return np.linalg.lstsq(X, y, rcond=None)[0]
def bootgo(y, dn, sg, extra=None, n=500):
    o = []
    for _ in range(n): i = rb.integers(0, len(y), len(y)); o.append(fitgo(y[i], dn[i], sg[i], None if extra is None else extra[i]))
    return np.percentile(np.array(o), [2.5, 97.5], axis=0)
e = np.where(EXT)[0]; dn_, dh_, sg_, cf_ = M.d_none.values[e], M.d_hist.values[e], sgn_all[e], CONF[e].astype(float)
b = fitgo(dh_, dn_, sg_); ci = bootgo(dh_, dn_, sg_)
say(f"\n[B4-1] behavior, {len(e)} extreme-count trials (conflict {int(cf_.sum())}, congruent {int((1 - cf_).sum())}): delta_hist = a*delta_none + c*sign(count) + b")
say(f"  gain a = {b[0]:.2f} [{ci[0][0]:.2f}, {ci[1][0]:.2f}] | offset c = {b[1]:+.2f} [{ci[0][1]:+.2f}, {ci[1][1]:+.2f}] | const {b[2]:+.2f}")
pred = np.c_[dn_, sg_, np.ones(len(e))] @ b
for nm, m in (("conflict", cf_ == 1), ("congruent", cf_ == 0)):
    r_ = dh_[m] - pred[m]; bs = [r_[rb.integers(0, len(r_), len(r_))].mean() for _ in range(500)]
    say(f"  {nm:9s} n={int(m.sum()):3d}: residual of the common rule {r_.mean():+.2f} [{np.percentile(bs, 2.5):+.2f}, {np.percentile(bs, 97.5):+.2f}] | sign-aligned history shift {(sg_[m] * (dh_[m] - dn_[m])).mean():+.2f}")
bx = fitgo(dh_, dn_, sg_, cf_); cix = bootgo(dh_, dn_, sg_, cf_)
say(f"  extra 'conflict' term added to the rule: {bx[3]:+.2f} [{cix[0][3]:+.2f}, {cix[1][3]:+.2f}]  (0 = conflict trials follow the same rule)")
for nm, m in (("conflict", cf_ == 1), ("congruent", cf_ == 0)):
    bm = fitgo(dh_[m], dn_[m], sg_[m]); cim = bootgo(dh_[m], dn_[m], sg_[m]); say(f"  rule fitted on {nm} only: gain {bm[0]:.2f} [{cim[0][0]:.2f}, {cim[1][0]:.2f}], offset {bm[1]:+.2f} [{cim[0][1]:+.2f}, {cim[1][1]:+.2f}]")
hi_ = M.contrast.values[e] >= 0.09; tg = np.where(M.side.values[e] == "right", 1, -1)
for nm, m in (("conflict", cf_ == 1), ("congruent", cf_ == 0)):
    mm = m & hi_; say(f"  accuracy at contrast >= 0.09, {nm} (n={int(mm.sum())}): no history {((dn_[mm] * tg[mm]) > 0).mean():.2f} -> history {((dh_[mm] * tg[mm]) > 0).mean():.2f}")

In [ ]:
# ===================== analysis 2: gain and offset carried by the last-position state, per layer =====================
P = PT.copy(); ex = P[(P.nr >= 8) | (P.nr <= 2)]; sg = np.where(ex.nr >= 8, 1.0, -1.0); dn = ex.d_none.values
say(f"\n[B4-2] patched rows: {len(ex)} trials (conflict {int(ex.conflict.sum())}, congruent {int((~ex.conflict).sum())}); the common rule fitted to these trials' own history runs:")
bt = fitgo(ex.d_hist.values, dn, sg); cit = bootgo(ex.d_hist.values, dn, sg); say(f"  target (history run): gain {bt[0]:.2f} [{cit[0][0]:.2f}, {cit[1][0]:.2f}], offset {bt[1]:+.2f} [{cit[0][1]:+.2f}, {cit[1][1]:+.2f}]")
say("per layer: gain / offset of delta_patched = a*delta_none + c*sign + b.  fwd = history state put into a no-history run (starts at gain 1, offset 0; the target is the line above); rev = no-history state put into a history run (starts at the target)")
rows_g = []
for L in PATCH_LAYERS:
    rec = dict(layer=L)
    for dr in ("fwd", "rev"):
        y = ex[f"{dr}_{L}"].values; bb = fitgo(y, dn, sg); cc = bootgo(y, dn, sg, n=300)
        rec.update({f"{dr}_gain": bb[0], f"{dr}_gain_lo": cc[0][0], f"{dr}_gain_hi": cc[1][0], f"{dr}_off": bb[1], f"{dr}_off_lo": cc[0][1], f"{dr}_off_hi": cc[1][1]})
    rows_g.append(rec)
G = pd.DataFrame(rows_g); G.to_csv(f"{OUT}/b4_gain_offset_by_layer.csv", index=False)
say(G[["layer", "fwd_gain", "fwd_gain_lo", "fwd_gain_hi", "fwd_off", "fwd_off_lo", "fwd_off_hi", "rev_gain", "rev_off"]].round(2).to_string(index=False))
# fraction of each component that has arrived (fwd): gain (1 - a_L)/(1 - a_target), offset c_L / c_target
G["gain_arrived"] = (1 - G.fwd_gain) / (1 - bt[0]); G["offset_arrived"] = G.fwd_off / bt[1]
say("\nfraction of the history effect that has arrived at the last position (fwd): gain loss (1 - a_L)/(1 - a_target) and offset c_L/c_target")
say(G[["layer", "gain_arrived", "offset_arrived"]].round(2).to_string(index=False))
# conflict vs congruent: same decomposition, separately
say("\nconflict vs congruent separately (fwd): gain / offset per layer")
out = {}
for nm, m in (("conflict", ex.conflict.values), ("congruent", ~ex.conflict.values)):
    if m.sum() < 8: continue
    rr = []
    for L in PATCH_LAYERS: bb = fitgo(ex[f"fwd_{L}"].values[m], dn[m], sg[m]); rr.append((L, bb[0], bb[1]))
    out[nm] = pd.DataFrame(rr, columns=["layer", "gain", "offset"]); bm = fitgo(ex.d_hist.values[m], dn[m], sg[m]); say(f"  {nm} (n={int(m.sum())}) target: gain {bm[0]:.2f}, offset {bm[1]:+.2f}"); say(out[nm].round(2).to_string(index=False))
# slope of the total history effect (all trials), conflict vs congruent
say("\ntotal history effect carried (slope of fwd - d_none on hist - none), conflict vs congruent")
for nm, m in (("conflict", ex.conflict.values), ("congruent", ~ex.conflict.values)):
    if m.sum() < 8: continue
    tot = (ex.d_hist - ex.d_none).values[m]; sl = {L: float(((ex[f"fwd_{L}"] - ex.d_none).values[m] * tot).sum() / (tot * tot).sum()) for L in PATCH_LAYERS}
    say(f"  {nm}: " + " ".join(f"L{L}:{v:.2f}" for L, v in sl.items()))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].errorbar(G.layer, G.fwd_gain, yerr=[G.fwd_gain - G.fwd_gain_lo, G.fwd_gain_hi - G.fwd_gain], marker="o", label="gain a_L (fwd)"); ax[0].axhline(bt[0], c="C0", ls="--", lw=.8); ax[0].axhline(1, c="gray", lw=.5)
ax[0].set_title("gain of the evidence carried by the patched state"); ax[0].set_xlabel("layer"); ax[0].legend(fontsize=8)
ax[1].errorbar(G.layer, G.fwd_off, yerr=[G.fwd_off - G.fwd_off_lo, G.fwd_off_hi - G.fwd_off], marker="s", color="C1", label="offset c_L (fwd)"); ax[1].axhline(bt[1], c="C1", ls="--", lw=.8); ax[1].axhline(0, c="gray", lw=.5)
ax[1].set_title("prior offset carried by the patched state"); ax[1].set_xlabel("layer"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/b4_gain_offset.png", dpi=130); plt.close()

In [ ]:
# ===================== SUMMARY: paste everything printed by the previous cells back to Claude =====================
open(f"{OUT}/B4_summary.txt", "w").write("\n".join(SUMMARY)); maybe_push('final', force=True)
with zipfile.ZipFile(f"{OUT}.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for f in os.listdir(OUT): zf.write(f"{OUT}/{f}", f)
print(f"zip written: {OUT}.zip")